# O Problema
Sliding Puzzle - Bloco Deslizante

In [1]:
# !wget -qq https://miro.medium.com/max/700/1*W7jg4GmEjGBypd9WPktasQ.gif
from IPython.display import Image
Image(url='https://miro.medium.com/max/700/1*W7jg4GmEjGBypd9WPktasQ.gif',width=200)

Anotações sobre a estrutura do quebra-cabeças:

---

###Nó
Estrutura de dados que mantém informações sobre:
- estado
- nó pai
- ação que o gerou
- custo acumulado

Expandir um Nó: Gerar nós sucessores, aplicando as ações permitidas ao estado que ele representa

Fronteira: Conjunto de nós descobertos que ainda aguardam ser examinados

Abordagem: Inicialmente, a fronteira possui apenas o estado inicial
Loop:
- Se a fronteira está vazia, então não temos solução
- Se não, removemos o nó da fronteira
- Se o nó contém o estado objetivo, retornamos a solução
- Se não, expandimos o nó e adicionamos os resultados a fronteira

---

###Estado
Uma descrição da situação do problema, com as informações relevantes para determinar as ações possíveis e seus resultados

---

In [2]:
class No:
    def __init__(self, estado, no_pai=None, acao=None, custo_acumulado=0):
      self.estado = estado
      self.no_pai = no_pai
      self.acao = acao
      self.custo_acumulado = custo_acumulado

    def __repr__(self):
      return f"No(estado={self.estado}, acao={self.acao}, custo={self.custo_acumulado})"

estado_inicial = ((1, 2, 3), (4, 5, 6), (7, 0, 8)) #0 é esp. vazio

no_raiz = No(estado=estado_inicial)
print("Nó raiz:")
display(no_raiz)

novo_estado = ((1, 2, 3), (4, 5, 6), (0, 7, 8)) #moveu p direita
no_sucessor = No(estado=novo_estado, no_pai=no_raiz, acao="mover_para_direita", custo_acumulado=no_raiz.custo_acumulado + 1)
print("\nNó sucessor:")
display(no_sucessor)

no_pai = no_sucessor.no_pai
print("\nNó pai do nó sucessor (nó raiz):")
display(no_pai)

Nó raiz:


No(estado=((1, 2, 3), (4, 5, 6), (7, 0, 8)), acao=None, custo=0)


Nó sucessor:


No(estado=((1, 2, 3), (4, 5, 6), (0, 7, 8)), acao=mover_para_direita, custo=1)


Nó pai do nó sucessor (nó raiz):


No(estado=((1, 2, 3), (4, 5, 6), (7, 0, 8)), acao=None, custo=0)

In [3]:
import collections

# Definir o estado objetivo para um puzzle 3x3
estado_objetivo = ((1, 2, 3), (4, 5, 6), (7, 8, 0))

def encontrar_posicao_vazia(estado):
    """Encontra a posição (linha, coluna) do espaço vazio (0) no estado."""
    for r in range(3):
        for c in range(3):
            if estado[r][c] == 0:
                return r, c
    return None

def obter_acoes_validas(estado):
    """Retorna uma lista de ações válidas ('cima', 'baixo', 'esquerda', 'direita')
       para o estado dado, com base na posição do espaço vazio."""
    r_vazio, c_vazio = encontrar_posicao_vazia(estado)
    acoes = []
    # Mover para cima (blank moves down)
    if r_vazio > 0: acoes.append('cima')
    # Mover para baixo (blank moves up)
    if r_vazio < 2: acoes.append('baixo')
    # Mover para esquerda (blank moves right)
    if c_vazio > 0: acoes.append('esquerda')
    # Mover para direita (blank moves left)
    if c_vazio < 2: acoes.append('direita')
    return acoes

def aplicar_acao(estado, acao):
    """Retorna o novo estado após aplicar a ação dada."""
    r_vazio, c_vazio = encontrar_posicao_vazia(estado)
    novo_estado = [list(row) for row in estado] # Converter para lista de listas para mutabilidade

    # Coordenadas para troca
    r_troca, c_troca = r_vazio, c_vazio
    if acao == 'cima': r_troca -= 1
    elif acao == 'baixo': r_troca += 1
    elif acao == 'esquerda': c_troca -= 1
    elif acao == 'direita': c_troca += 1

    # Realizar a troca
    novo_estado[r_vazio][c_vazio], novo_estado[r_troca][c_troca] = \
        novo_estado[r_troca][c_troca], novo_estado[r_vazio][c_vazio]

    return tuple(tuple(row) for row in novo_estado) # Converter de volta para tupla de tuplas

def reconstruir_caminho(no_final):
    """Reconstrói o caminho da solução do nó final até o nó raiz."""
    caminho = []
    no_atual = no_final
    while no_atual.no_pai is not None:
        caminho.append(no_atual.acao)
        no_atual = no_atual.no_pai
    return caminho[::-1] # Inverter a lista para obter o caminho do início ao fim

# Resolver o quebra-cabeças usando Buscas

## Busca em largura

In [4]:
def bfs(estado_inicial, estado_objetivo):
  fronteira = collections.deque()
  no_raiz = No(estado_inicial)
  fronteira.append(no_raiz)

  visitados= {estado_inicial}

  while fronteira:
    no_atual = fronteira.popleft()
    if no_atual.estado == estado_objetivo:
      return reconstruir_caminho(no_atual), no_atual.custo_acumulado

    #expande o nó atual
    for acao in obter_acoes_validas(no_atual.estado):
      proximo_estado = aplicar_acao(no_atual.estado, acao)
      #se prox estado nao foi visitado, adiciona a fronteira e marca como visitado
      if proximo_estado not in visitados:
        visitados.add(proximo_estado)
        proximo_no = No(proximo_estado, no_pai=no_atual, acao=acao, custo_acumulado=no_atual.custo_acumulado + 1)
        fronteira.append(proximo_no)

  return None, None

### Exemplo de Uso do BFS

In [5]:
# Definir um estado inicial diferente (apenas para demonstração)
estado_teste_inicial = ((1, 2, 3), (4, 0, 6), (7, 5, 8))

print(f"Estado Inicial para o teste:\n{estado_teste_inicial}")
print(f"Estado Objetivo:\n{estado_objetivo}")

solucao_caminho, custo_solucao = bfs(estado_teste_inicial, estado_objetivo)

if solucao_caminho:
    print(f"\nSolução encontrada em {custo_solucao} passos:")
    print(solucao_caminho)
else:
    print("\nNenhuma solução encontrada para o estado inicial fornecido.")


Estado Inicial para o teste:
((1, 2, 3), (4, 0, 6), (7, 5, 8))
Estado Objetivo:
((1, 2, 3), (4, 5, 6), (7, 8, 0))

Solução encontrada em 2 passos:
['baixo', 'direita']


## Busca em Profundidade

In [8]:
def dfs(estado_inicial, estado_objetivo):
    # fronteira se comporta como uma pilha Last In, First Out
    # por isso usaremos uma lista Python e os métodos append/pop
    fronteira = []
    no_raiz = No(estado_inicial)
    fronteira.append(no_raiz)

    visitados = {estado_inicial} # estados visitados

    while fronteira:
        no_atual = fronteira.pop()

        # Verifica se é o objetivo
        if no_atual.estado == estado_objetivo:
            return reconstruir_caminho(no_atual), no_atual.custo_acumulado

        # Expande o nó
        for acao in obter_acoes_validas(no_atual.estado):
            proximo_estado = aplicar_acao(no_atual.estado, acao)

            # Se o próximo estado não foi visitado, adiciona-o à fronteira e marca como visitado
            if proximo_estado not in visitados:
                visitados.add(proximo_estado)
                proximo_no = No(proximo_estado, no_pai=no_atual, acao=acao, custo_acumulado=no_atual.custo_acumulado + 1)
                fronteira.append(proximo_no)

    return None, None # Se nenhuma solução for encontrada




### Exemplo de Uso do DFS

In [10]:

estado_teste_inicial_dfs = ((1, 2, 3), (4, 0, 6), (7, 5, 8))

print(f"Estado Inicial para o teste:\n{estado_teste_inicial_dfs}")
print(f"Estado Objetivo:\n{estado_objetivo}")

solucao_caminho_dfs, custo_solucao_dfs = dfs(estado_teste_inicial_dfs, estado_objetivo)

if solucao_caminho_dfs:
    print(f"\nSolução encontrada em {custo_solucao_dfs} passos:")
    print(solucao_caminho_dfs)
else:
    print("\nNenhuma solução encontrada para o estado inicial fornecido com DFS.")

Estado Inicial para o teste:
((1, 2, 3), (4, 0, 6), (7, 5, 8))
Estado Objetivo:
((1, 2, 3), (4, 5, 6), (7, 8, 0))

Solução encontrada em 49284 passos:
['direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'cima', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'cima', 'esquerda', 'esquerda', 'baixo', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', 'baixo', 'esquerda', 'cima', 'direita', 'baixo', 'esquerda', 'esquerda', 'cima', 'direita', 'direita', '

## Discorra sobre o desempenho dos métodos em questões de:

###1.   Consumo de memória:
O consumo de memória usando o algoritmo BFS é muito maior, pois ele armazena todos os nós de um mesmo nível (fronteira) antes de descer para o próximo. Isso quer dizer que, quanto mais profunda estiver a solução, o consumo de memória vai crescendo exponencialmente. Nesse sentido, o DFS consome muito menos, ao armazenar somente o caminho da busca.

###2.   Processamento:
Em termos de processamento, o BFS explora os estados nível por nível e, em alguns casos, o algoritmo pode precisar explorar uma grande quantidade de estados até encontrar a solução. Entretanto, ele se mostra muito mais eficiente em encontrar a solução com o menor número de movimentos possível. Dependendo do caso, o DFS pode encontrar uma solução mais rápido, mas não garante que ela seja mais curta.

